# Visualisierung eines grundlegenden Gradientenabstiegs

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

Bei der Verwendung des Gradientenabstiegs können wir auf einige Probleme stoßen:

- In einem lokalen Minimum gefangen werden.

- Überschießen und das globale Optimum verfehlen. Dies ist ein direktes Ergebnis davon, dass man sich zu schnell entlang der Gradientenrichtung bewegt.

- Oszillation: Dies ist ein Phänomen, das auftritt, wenn sich der Funktionswert nicht wesentlich ändert, unabhängig davon, in welche Richtung er sich bewegt.

Um diese Probleme zu bekämpfen, wird ein Momentum-Term $ \alpha $ verwendet, um die Lernrate zu stabilisieren, wenn man sich auf den globalen Optimalwert zubewegt.

Wir definieren nun eine `gradient_descent()` Funktion. In dieser Funktion endet die Schleife, wenn entweder:

- Die Anzahl der Iterationen einen Maximalwert überschreitet.

- Der Unterschied in den Funktionswerten zwischen zwei aufeinanderfolgenden Iterationen unter einen bestimmten Schwellenwert fällt.

In [ ]:
def gradient_descent(max_iterations, threshold, b_init,
                     obj_func, grad_func,
                     learning_rate=0.05, momentum=0.8):
    ''' 
    Input:
    - max_iterations  : Maximum number of iterations to run
    - threshold       : Stop if the difference in function values between two successive iterations falls below this threshold
    - b_init          : Initial point from where to start gradient descent
    - obj_func        : Reference to the function that computes the objective function
    - grad_func       : Reference to the function that computes the gradient of the function
    - learning_rate   : Step size for gradient descent. It should be in [0,1]
    - momentum        : Momentum to use. It should be in [0,1]
    Output:
    - b_history       : All points in space, visited by gradient descent at which the objective function was evaluated
    - f_history       : Corresponding value of the objective function computed at each point
    '''
    
    b = b_init
    b_history = b
    f_history = obj_func(b)
    delta_b = np.zeros(b.shape)
    i = 0
    diff = 1.0e10
   
    while  i<max_iterations and diff>threshold:
        delta_b = -learning_rate*grad_func(b) + momentum*delta_b
        b = b + delta_b
        
        # store the history of w and f
        b_history = np.vstack((b_history, b))
        f_history = np.vstack((f_history, obj_func(b)))
        
        # update iteration number and diff between successive values
        # of objective function
        i += 1
        diff = np.absolute(f_history[-1] - f_history[-2])
    
    return b_history, f_history

Wir können nun diese allgemeine Implementierung des Gradientenabstiegs auf eine beispielhafte 2D-Funktion $ f(b_1 , b_2) = b_1^2 + b_2^2$ mit kreisförmigen Konturen anwenden.

Die Funktion hat einen Minimalwert von Null am Ursprung. Lassen Sie uns die Funktion zunächst visualisieren und dann ihren Minimalwert mit dem Gradientenabstieg finden.

Die ```visualize_fw()``` Funktion unten generiert 2500 gleichmäßig verteilte Punkte auf einem Gitter und berechnet den Funktionswert an jedem Punkt.

Die ```function_plot()``` Funktion zeigt alle Punkte in verschiedenen Farben an, abhängig vom Wert von $f(b_1 , b_2)$ an diesem Punkt. Alle Punkte, an denen der Funktionswert gleich ist, haben die gleiche Farbe:

In [ ]:
def visualize_fw():
    xcoord = np.linspace(-10.0, 10.0, 50)
    ycoord = np.linspace(-10.0, 10.0, 50)
    b_1, b_2 = np.meshgrid(xcoord, ycoord)
    pts = np.vstack((b_1.flatten(), b_2.flatten()))
    
    # All 2D points on the grid
    pts = pts.transpose()
    
    # Function value at each point
    f_vals = np.sum(pts*pts, axis=1)
    function_plot(pts, f_vals)
    plt.title('Objective Function Shown in Color')
    plt.show()
    return pts, f_vals

# Helper function to annotate a single point
def annotate_pt(text, xy, xytext, color):
    plt.plot(xy[0], xy[1], marker='P', markersize=10, c=color)
    plt.annotate(text, xy=xy, xytext=xytext,
                 arrowprops=dict(arrowstyle="->",
                 color=color,
                 connectionstyle='arc3'))

# Plot the function
# Pts are 2D points and f_val is the corresponding function value
def function_plot(pts, f_val):
    f_plot = plt.scatter(pts[:,0], pts[:,1],
                         c=f_val, vmin=min(f_val), vmax=max(f_val),
                         cmap='RdBu_r')
    plt.colorbar(f_plot)
    # Show the optimal point
    annotate_pt('global minimum', (0,0), (-5,-7), 'yellow')    

pts, f_vals = visualize_fw()

Jetzt verwenden wir die Gradientenabstiegsfunktion, um unsere Zielfunktion zu minimieren. Um `gradient_descent()` aufzurufen, definieren wir zwei Funktionen:

`f()`: Berechnet die Zielfunktion an jedem Punkt `b`

`grad()`: Berechnet den Gradienten an jedem Punkt `b`

Um die Auswirkungen verschiedener Hyperparameter auf den Gradientenabstieg zu verstehen, ruft die Funktion `solve_fw()` `gradient_descent()` mit 5 Iterationen für verschiedene Werte von Lernrate und Momentum auf.

## Was ist Momentum?

SGD hat Schwierigkeiten, in Schluchten zu navigieren, d.h. in Bereichen, in denen sich die Oberfläche in einer Dimension viel steiler krümmt als in einer anderen, die um lokale Optima herum häufig vorkommen. In diesen Szenarien oszilliert SGD über die Hänge der Schlucht, während es nur zögerliche Fortschritte am Boden in Richtung des lokalen Optimums macht.

<img src="./assets/no-momentum.gif" alt="https://mlfromscratch.com/optimizers-explained/#/" width="500"/>
<img src="./assets/momentum.gif" alt="https://mlfromscratch.com/optimizers-explained/#/" width="500"/>

Momentum ist eine Methode, die dabei hilft, SGD in die relevante Richtung zu beschleunigen und Oszillationen zu dämpfen, wie in den obigen Bildern zu sehen ist. Dies geschieht, indem ein Bruchteil $ \gamma $ des Update-Vektors des vorherigen Zeitschritts zum aktuellen Update-Vektor hinzugefügt wird:

$$ v_{t} = \gamma v_{t-1} + \alpha \frac{\partial}{\partial b} J(b)  $$
$$  b_{t} = b_{t-1} - v_{t} $$

*Hinweis: Einige Implementierungen tauschen die Vorzeichen in den Gleichungen aus. Der Momentum-Term $ \gamma $ wird normalerweise auf einen Wert zwischen 0 und 1 gesetzt.*

Im Wesentlichen stoßen wir beim Verwenden von Momentum einen Ball einen Hügel hinunter. Der Ball sammelt Momentum, während er den Hügel hinunterrollt und wird immer schneller auf dem Weg (bis er seine Endgeschwindigkeit erreicht, wenn es Luftwiderstand gibt, d.h. $ \gamma < 1 $). Das Gleiche passiert mit unseren Parameteraktualisierungen: Der Momentum-Term erhöht sich für Dimensionen, deren Gradienten in die gleiche Richtung zeigen, und reduziert Aktualisierungen für Dimensionen, deren Gradienten ihre Richtung ändern. Als Ergebnis erzielen wir eine schnellere Konvergenz und reduzierte Oszillation.

Wenn Sie mit Momentum und Lernrate spielen möchten, empfehle ich den Besuch der Seite von distill für [Why Momentum Really Work](https://distill.pub/2017/momentum/).

Die Funktion `visualize_learning()` plottet die Werte von $ (b_1 , b_2) $, wobei Funktionswerte in verschiedenen Farben angezeigt werden. Die Pfeile im Plot erleichtern es nachzuverfolgen, welcher Punkt vom letzten aktualisiert wurde:

In [ ]:
# Objective function
def f(b):
    return np.sum(b*b)

# Function to compute the gradient
def grad(b):
    return 2*b

# Function to plot the objective function
# and learning history annotated by arrows
# to show how learning proceeded
def visualize_learning(b_history):  
    
    # Make the function plot
    function_plot(pts, f_vals)
    
    # Plot the history
    plt.plot(b_history[:,0], b_history[:,1], marker='o', c='magenta') 
    
    # Annotate the point found at last iteration
    annotate_pt('minimum found',
                (b_history[-1,0], b_history[-1,1]),
                (-1,7), 'green')
    iter = b_history.shape[0]
    for b, i in zip(b_history, range(iter-1)):
        # Annotate with arrows to show history
        plt.annotate("",
                    xy=b, xycoords='data',
                    xytext=b_history[i+1,:], textcoords='data',
                    arrowprops=dict(arrowstyle='<-',
                            connectionstyle='angle3'))     
    
def solve_fw():
    # Setting up
    rand = np.random.RandomState(19)
    b_init = rand.uniform(-10,10,2)
    fig, ax = plt.subplots(nrows=4, ncols=5, figsize=(18, 12))
    learning_rates = [0.05, 0.2, 0.5, 0.8, 1]
    momentum = [0, 0.2, 0.5, 0.9]
    ind = 1
    
    # Iteration through all possible parameter combinations
    for alpha in momentum:
        for eta, col in zip(learning_rates, [0, 1, 2, 3, 4]):
            plt.subplot(4, 5, ind)        
            b_history, f_history = gradient_descent(5, -1, b_init, f, grad, eta, alpha)
            
            visualize_learning(b_history)
            ind = ind+1
            plt.text(-9, 12, 'Learning Rate = ' + str(eta), fontsize=13)
            if col==1:
                plt.text(10, 15, 'momentum = ' + str(alpha), fontsize=20)

    fig.subplots_adjust(hspace=0.5, wspace=.3)
    plt.show()

Lassen Sie uns ```solve_fw()``` ausführen und sehen, wie sich Lernrate und Momentum auf den Gradientenabstieg auswirken:

In [ ]:
solve_fw()

Dieses Beispiel verdeutlicht die Rolle von sowohl Momentum als auch Lernrate.

Im ersten Plot, mit null Momentum und Lernrate auf 0,05 gesetzt, ist das Lernen langsam und der Algorithmus erreicht nicht das globale Minimum. Eine Erhöhung des Momentums beschleunigt das Lernen, wie wir aus den Plots in der ersten Spalte sehen können. Das andere Extrem ist die letzte Spalte, in der die Lernrate hoch gehalten wird. Dies verursacht Oszillationen, die bis zu einem gewissen Grad durch Hinzufügen von Momentum kontrolliert werden können.

Die allgemeine Richtlinie für den Gradientenabstieg besteht darin, kleine Werte der Lernrate und höhere Werte des Momentums zu verwenden.